# Does the Value of Non-Parent Backtracking Depend on Task Structure?

**Research question 4:** *Is the benefit of non-parent backtracking greater on tasks with explicit dependency structure between reasoning steps (Mini Crosswords) than on tasks with shorter, more independent reasoning chains (Game of 24)?*

This notebook does **not** pool the two benchmarks into a single statistical test.
 Game24 and Mini Crosswords differ in outcome definition (binary solve vs. continuous partial credit), search implementation,candidate-generation structure, pruning mechanism, and sample size (n=100 vs n=20)
Instead, RQ4 is answered as a **cross-benchmark comparison of within-benchmark effects**, in three parts:

This notebook is self-contained and independently reproducible.

In [5]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon, rankdata
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from itertools import combinations

pd.set_option("display.width", 120)
CONDITIONS = ["A", "B", "C", "D"]

In [6]:
#load the raw JSON llogs from both the experiments to perform checks on thm.
BASE = Path("../logs/final_experiment")

game24_files = {
    "A": BASE / "game24/A_parent/game24_A_parent_bedrock_B3_E1_budget50_vth0.5_900_1000_20260817_184855.json",
    "B": BASE / "game24/B_beta_c/game24_B_beta_c_bedrock_B3_E1_budget50_vth0.5_900_1000_20260817_223127.json",
    "C": BASE / "game24/C_fixed_k2/game24_C_fixed_k2_bedrock_B3_E1_budget50_vth0.5_900_1000_20260818_003244.json",
    "D": BASE / "game24/D_strict_nonparent/game24_D_strict_nonparent_bedrock_B3_E1_budget50_vth0.5_900_1000_20260820_010537.json",
}

crossword_files = {
    "A": BASE / "crosswords/A_parent/crosswords_A_parent_bedrock_B5_E1_budget50_vth0.5_maxstate3_pruneTrue_0_20_20260820_120255.json",
    "B": BASE / "crosswords/B_beta_c/crosswords_B_beta_c_bedrock_B5_E1_budget50_vth0.5_maxstate3_pruneTrue_0_20_20260822_104719.json",
    "C": BASE / "crosswords/C_fixed_k2/crosswords_C_fixed_k2_bedrock_B5_E1_budget50_vth0.5_maxstate3_pruneTrue_0_20_20260821_131200.json",
    "D": BASE / "crosswords/D_strict_nonparent/crosswords_D_strict_nonparent_bedrock_B5_E1_budget50_vth0.5_maxstate3_pruneTrue_0_20_20260822_221611.json",
}

game24_raw    = {c: json.load(open(p)) for c, p in game24_files.items()}
crossword_raw = {c: json.load(open(p)) for c, p in crossword_files.items()}

print("Game24 puzzle counts:   ", {c: len(v) for c, v in game24_raw.items()})
print("Crosswords puzzle counts:", {c: len(v) for c, v in crossword_raw.items()})

# check the pairing, whehter same puzzle set across conditions, within each benchmark
for name, raw in [("Game24", game24_raw), ("Crosswords", crossword_raw)]:
    id_sets = {c: {r["idx"] for r in recs} for c, recs in raw.items()}
    ok = all(id_sets["A"] == id_sets[c] for c in CONDITIONS)
    print(f"{name}: identical puzzle set across A-D? {ok}")

Game24 puzzle counts:    {'A': 100, 'B': 100, 'C': 100, 'D': 100}
Crosswords puzzle counts: {'A': 20, 'B': 20, 'C': 20, 'D': 20}
Game24: identical puzzle set across A-D? True
Crosswords: identical puzzle set across A-D? True


In [7]:
# Build one row-per-puzzle dataframe per benchmark


def extract_game24_row(r, condition):
    return {
        "puzzle_id":             r["idx"],
        "condition":             condition,
        "solved":                r["solution"] is not None,
        "nodes_explored":        r["nodes_explored"],
        "candidates_evaluated":  r["candidates_evaluated"],
        "candidates_pruned":     r["candidates_pruned"],
    }

def extract_crossword_row(r, condition):
    info = (r.get("infos") or [{}])[0]
    return {
        "puzzle_id":             r["idx"],
        "condition":             condition,
        "r_word":                info.get("r_word"),
        "r_letter":              info.get("r_letter"),
        "r_game":                bool(info.get("r_game")),
        "nodes_explored":        r["nodes_explored"],
        "candidates_evaluated":  r["candidates_evaluated"],
        "candidates_pruned":     r["candidates_pruned"],
    }

df_g24 = pd.concat(
    [pd.DataFrame([extract_game24_row(r, c) for r in game24_raw[c]]) for c in CONDITIONS],
    ignore_index=True
)

df_cw = pd.concat(
    [pd.DataFrame([extract_crossword_row(r, c) for r in crossword_raw[c]]) for c in CONDITIONS],
    ignore_index=True
)

print("df_g24:", df_g24.shape, "| df_cw:", df_cw.shape)

df_g24: (400, 6) | df_cw: (80, 8)


In [8]:
# rank-biserial correlation for paired samples, as a measure of effect size for Wilcoxon signed-rank test

def rank_biserial(x, y):
    diff = np.array(x) - np.array(y)
    diff = diff[diff != 0]
    if len(diff) == 0:
        return 0.0
    ranks = rankdata(np.abs(diff))
    pos = ranks[diff > 0].sum()
    neg = ranks[diff < 0].sum()
    return (pos - neg) / (pos + neg)

def friedman_then_posthoc(df, metric, conditions=CONDITIONS):
    wide = df.pivot(index="puzzle_id", columns="condition", values=metric)[conditions].dropna()
    friedman_result = friedmanchisquare(*[wide[c] for c in conditions])

    result = {
        "metric": metric,
        "n_paired": len(wide),
        "friedman_chi2": friedman_result.statistic,
        "friedman_p": friedman_result.pvalue,
        "pairwise": None,
    }

    if friedman_result.pvalue >= 0.05:
        return result

    rows = []
    for c1, c2 in combinations(conditions, 2):
        stat, p = wilcoxon(wide[c1], wide[c2])
        rows.append({
            "comparison": f"{c1} vs {c2}",
            "median_diff": (wide[c1] - wide[c2]).median(),
            "W": stat, "p_value": p,
            "rank_biserial": rank_biserial(wide[c1], wide[c2]),
        })
    pairwise = pd.DataFrame(rows)
    pairwise["p_holm"] = multipletests(pairwise["p_value"], method="holm")[1]
    pairwise["significant"] = pairwise["p_holm"] < 0.05
    result["pairwise"] = pairwise.sort_values("p_holm").reset_index(drop=True)
    return result

## Part 1 — Task performance: the A vs B effect on each benchmark's own outcome

Game24's outcome is binary (solved/unsolved, n=100); Crosswords' primary outcome is continuous partial credit (`r_word`, n=20), with `r_letter` as a secondary measure. These are reported side by side **as different measures on different scales**, not converted to a common unit — a +7 percentage-point solve-rate change and a +0.035 mean `r_word` change are not directly comparable magnitudes.

In [ ]:
# Game24: A vs B, solve rate 

g24_wide_solved = df_g24.pivot(index="puzzle_id", columns="condition", values="solved")[CONDITIONS]
g24_solve_rate = g24_wide_solved.mean() * 100

g24_contingency = pd.crosstab(
    g24_wide_solved["A"].map({True: "A solved", False: "A unsolved"}),
    g24_wide_solved["B"].map({True: "B solved", False: "B unsolved"})
).reindex(index=["A solved", "A unsolved"], columns=["B solved", "B unsolved"], fill_value=0)

g24_mcnemar = mcnemar(g24_contingency.values, exact=True)

print("Game24 solve rate — A:", round(g24_solve_rate['A'], 1), "%  B:", round(g24_solve_rate['B'], 1), "%")
print(f"Difference (B-A): {g24_solve_rate['B'] - g24_solve_rate['A']:+.1f} pp")
print(f"McNemar exact p (single pre-registered A-vs-B test, uncorrected): {g24_mcnemar.pvalue:.4f}")

Game24 solve rate — A: 92.0 %  B: 99.0 %
Difference (B-A): +7.0 pp
McNemar exact p (single pre-registered A-vs-B test, uncorrected): 0.0156


In [10]:


cw_rword_result   = friedman_then_posthoc(df_cw, "r_word")
cw_rletter_result = friedman_then_posthoc(df_cw, "r_letter")

def ab_status(result, label):
    print(f"--- Crosswords {label}: 4-way Friedman p={result['friedman_p']:.4f} ---")
    if result["pairwise"] is None:
        print("Omnibus not significant -> A vs B not tested to significance via this pipeline.")
        return None
    row = result["pairwise"][result["pairwise"]["comparison"] == "A vs B"]
    if row.empty:
        print("A vs B not found in pairwise table (unexpected).")
        return None
    print(row.to_string(index=False))
    return row.iloc[0]

rword_ab   = ab_status(cw_rword_result, "r_word")
rletter_ab = ab_status(cw_rletter_result, "r_letter")

cw_means = df_cw.groupby("condition")[["r_word", "r_letter"]].mean().reindex(CONDITIONS)
print("\nMeans by condition:")
print(cw_means.round(3))

--- Crosswords r_word: 4-way Friedman p=0.1193 ---
Omnibus not significant -> A vs B not tested to significance via this pipeline.
--- Crosswords r_letter: 4-way Friedman p=0.0429 ---
comparison  median_diff    W  p_value  rank_biserial   p_holm  significant
    A vs B        -0.04 68.5 0.458551       -0.19883 0.917102        False

Means by condition:
           r_word  r_letter
condition                  
A           0.355     0.518
B           0.390     0.604
C           0.240     0.388
D           0.300     0.468


In [11]:

# Part 1 summary table


part1_summary = pd.DataFrame([
    {
        "benchmark": "Game24", "outcome": "solve rate (%)",
        "A": round(g24_solve_rate["A"], 1), "B": round(g24_solve_rate["B"], 1),
        "diff (B-A)": round(g24_solve_rate["B"] - g24_solve_rate["A"], 1),
        "test": "McNemar exact (pre-registered single test)",
        "p_value": round(g24_mcnemar.pvalue, 4),
        "significant": g24_mcnemar.pvalue < 0.05,
    },
    {
        "benchmark": "Crosswords", "outcome": "mean r_word",
        "A": round(cw_means.loc["A", "r_word"], 3), "B": round(cw_means.loc["B", "r_word"], 3),
        "diff (B-A)": round(cw_means.loc["B", "r_word"] - cw_means.loc["A", "r_word"], 3),
        "test": "Friedman (4-way)",
        "p_value": round(cw_rword_result["friedman_p"], 4),
        "significant": cw_rword_result["friedman_p"] < 0.05,
    },
    {
        "benchmark": "Crosswords", "outcome": "mean r_letter",
        "A": round(cw_means.loc["A", "r_letter"], 3), "B": round(cw_means.loc["B", "r_letter"], 3),
        "diff (B-A)": round(cw_means.loc["B", "r_letter"] - cw_means.loc["A", "r_letter"], 3),
        "test": "Friedman (4-way); A-vs-B Holm p shown if pairwise ran",
        "p_value": round(cw_rletter_result["friedman_p"], 4),
        "significant": cw_rletter_result["friedman_p"] < 0.05,
    },
])

print("Note: rows use different outcome scales (percentage points vs. 0-1 partial credit).")
print("The 'diff' column is NOT directly comparable across rows.\n")
part1_summary

Note: rows use different outcome scales (percentage points vs. 0-1 partial credit).
The 'diff' column is NOT directly comparable across rows.



,benchmark,outcome,A,B,diff (B-A),test,p_value,significant
0,Game24,solve rate (%),92.000,99.000,7.000,McNemar exact (pre-registered single test),0.0156,True
1,Crosswords,mean r_word,0.355,0.390,0.035,Friedman (4-way),0.1193,False
2,Crosswords,mean r_letter,0.518,0.604,0.086,Friedman (4-way); A-vs-B Holm p shown if pairw...,0.0429,True


## Part 2 — Search efficiency: the A vs B (and A vs C/D) effect on how much search was required

For each benchmark: `nodes_explored`, `candidates_evaluated`, `candidates_pruned`. Game24's own analysis already established that none of the three efficiency omnibus tests were significant (nodes χ²=1.89, p=0.596; evaluated χ²=0.75, p=0.861; pruned χ²=2.77, p=0.428) — recomputed here for a self-contained record, then compared directly against the same three tests run on Crosswords.

In [12]:
efficiency_metrics = ["nodes_explored", "candidates_evaluated", "candidates_pruned"]

g24_efficiency_results = {m: friedman_then_posthoc(df_g24, m) for m in efficiency_metrics}
cw_efficiency_results  = {m: friedman_then_posthoc(df_cw, m) for m in efficiency_metrics}

efficiency_omnibus_comparison = pd.DataFrame([
    {
        "metric": m,
        "Game24_chi2": round(g24_efficiency_results[m]["friedman_chi2"], 3),
        "Game24_p": round(g24_efficiency_results[m]["friedman_p"], 4),
        "Game24_significant": g24_efficiency_results[m]["friedman_p"] < 0.05,
        "Crosswords_chi2": round(cw_efficiency_results[m]["friedman_chi2"], 3),
        "Crosswords_p": round(cw_efficiency_results[m]["friedman_p"], 4),
        "Crosswords_significant": cw_efficiency_results[m]["friedman_p"] < 0.05,
    }
    for m in efficiency_metrics
])

efficiency_omnibus_comparison

,metric,Game24_chi2,Game24_p,Game24_significant,Crosswords_chi2,Crosswords_p,Crosswords_significant
0,nodes_explored,1.886,0.5964,False,17.912,0.0005,True
1,candidates_evaluated,0.751,0.8612,False,3.874,0.2754,False
2,candidates_pruned,2.773,0.4280,False,4.264,0.2343,False


In [9]:
for m in efficiency_metrics:
    result = cw_efficiency_results[m]
    print(f"--- Crosswords {m}: Friedman p={result['friedman_p']:.4f} ---")
    if result["pairwise"] is None:
        print("Omnibus not significant -> no pairwise tests licensed.\n")
        continue
    relevant = result["pairwise"][result["pairwise"]["comparison"].isin(["A vs B", "A vs C", "A vs D"])]
    print(relevant.to_string(index=False))
    print()

# Same A-vs-B/C/D extraction for the accuracy metrics, so Part 3 can use
# A-vs-B-SPECIFIC significance rather than the 4-way omnibus alone.
for label, result in [("r_word", cw_rword_result), ("r_letter", cw_rletter_result)]:
    print(f"--- Crosswords {label}: Friedman p={result['friedman_p']:.4f} ---")
    if result["pairwise"] is None:
        print("Omnibus not significant -> no pairwise tests licensed.\n")
        continue
    relevant = result["pairwise"][result["pairwise"]["comparison"].isin(["A vs B", "A vs C", "A vs D"])]
    print(relevant.to_string(index=False))
    print()


--- Crosswords nodes_explored: Friedman p=0.0005 ---
comparison  median_diff    W  p_value  rank_biserial   p_holm  significant
    A vs D         14.5 11.5 0.002085       0.849673 0.012510         True
    A vs C         14.0 14.5 0.009701       0.758333 0.038804         True
    A vs B          0.0 39.5 0.413871       0.247619 0.454250        False

--- Crosswords candidates_evaluated: Friedman p=0.2754 ---
Omnibus not significant -> no pairwise tests licensed.

--- Crosswords candidates_pruned: Friedman p=0.2343 ---
Omnibus not significant -> no pairwise tests licensed.

--- Crosswords r_word: Friedman p=0.1193 ---
Omnibus not significant -> no pairwise tests licensed.

--- Crosswords r_letter: Friedman p=0.0429 ---
comparison  median_diff    W  p_value  rank_biserial   p_holm  significant
    A vs C         0.04 36.0 0.172675        0.40000 0.690700        False
    A vs D         0.04 40.5 0.267874        0.32500 0.803621        False
    A vs B        -0.04 68.5 0.458551       -0

In [10]:
# --- Descriptive means, both benchmarks, for context alongside the tests ---

g24_means_eff = df_g24.groupby("condition")[efficiency_metrics].mean().reindex(CONDITIONS).round(2)
cw_means_eff  = df_cw.groupby("condition")[efficiency_metrics].mean().reindex(CONDITIONS).round(2)

print("Game24 mean efficiency metrics by condition:")
print(g24_means_eff)
print("\nCrosswords mean efficiency metrics by condition:")
print(cw_means_eff)

Game24 mean efficiency metrics by condition:
           nodes_explored  candidates_evaluated  candidates_pruned
condition                                                         
A                   18.36                 41.73              23.37
B                   18.33                 42.38              24.05
C                   18.28                 45.49              27.21
D                   18.43                 42.09              23.66

Crosswords mean efficiency metrics by condition:
           nodes_explored  candidates_evaluated  candidates_pruned
condition                                                         
A                    32.7                 90.50              57.80
B                    28.1                110.55              82.45
C                    17.4                 81.20              63.80
D                    14.5                 79.55              65.05


## Part 3 Type of benefit: does each benchmark's effect show up as accuracy, efficiency, both, or neither?

This section only restates results already established in Parts 1-2, organised to answer RQ4 directly — it does not run any new test.

In [11]:
# Two significance levels are kept separate on purpose:
#   - "A vs B": does the model-selected recovery policy change the outcome?
#   - "A vs C/D": does non-parent backtracking in general change the outcome?
# A benchmark can show one without the other, as Crosswords does here.

def pairwise_flag(result, comparison):
    """True only if the omnibus licensed pairwise tests AND this specific
    comparison survived Holm correction. False (not None) if the omnibus
    was non-significant -- that IS a real 'no effect' finding, not missing data."""
    if result["pairwise"] is None:
        return False
    row = result["pairwise"][result["pairwise"]["comparison"] == comparison]
    return bool(row.iloc[0]["significant"]) if not row.empty else False

# --- Game24: single pre-registered A-vs-B test for accuracy; omnibus-only for efficiency ---
g24_accuracy_AvB   = g24_mcnemar.pvalue < 0.05
g24_efficiency_AvB = False  # all three efficiency omnibus tests were non-significant -> no pairwise licensed at all
g24_efficiency_broader = False
g24_accuracy_broader = None  # not applicable -- Game24 has no separate 'broader non-parent' accuracy test beyond A vs B/C/D, all under the same McNemar family

# --- Crosswords: pull A-vs-B specifically, and A-vs-C/A-vs-D as the 'broader non-parent' check ---
cw_accuracy_AvB = pairwise_flag(cw_rword_result, "A vs B") or pairwise_flag(cw_rletter_result, "A vs B")
cw_accuracy_broader = any(
    pairwise_flag(r, comp) for r in [cw_rword_result, cw_rletter_result] for comp in ["A vs C", "A vs D"]
)
cw_efficiency_AvB = any(pairwise_flag(cw_efficiency_results[m], "A vs B") for m in efficiency_metrics)
cw_efficiency_broader = any(
    pairwise_flag(cw_efficiency_results[m], comp) for m in efficiency_metrics for comp in ["A vs C", "A vs D"]
)

pattern_table = pd.DataFrame([
    {
        "benchmark": "Game24",
        "reasoning structure": "short, largely independent steps",
        "accuracy: A vs B significant?": g24_accuracy_AvB,
        "accuracy: broader non-parent (A vs C/D) significant?": "n/a - same McNemar family as A vs B, all non-significant post-Holm",
        "efficiency: A vs B significant?": g24_efficiency_AvB,
        "efficiency: broader non-parent (A vs C/D) significant?": g24_efficiency_broader,
    },
    {
        "benchmark": "Crosswords",
        "reasoning structure": "high inter-step dependency (crossing words)",
        "accuracy: A vs B significant?": cw_accuracy_AvB,
        "accuracy: broader non-parent (A vs C/D) significant?": cw_accuracy_broader,
        "efficiency: A vs B significant?": cw_efficiency_AvB,
        "efficiency: broader non-parent (A vs C/D) significant?": cw_efficiency_broader,
    },
])

pattern_table


,benchmark,reasoning structure,accuracy: A vs B significant?,accuracy: broader non-parent (A vs C/D) significant?,efficiency: A vs B significant?,efficiency: broader non-parent (A vs C/D) significant?
0,Game24,"short, largely independent steps",True,"n/a - same McNemar family as A vs B, all non-s...",False,False
1,Crosswords,high inter-step dependency (crossing words),False,False,False,True
